# 02 - Sample from trained DiT checkpoints and create visual grids

    This notebook loads the latent Imagenette checkpoints from the runs, samples latents with DDIM, decodes them through the VAE, and saves class grids, CFG sweeps, and sampling-step sweeps.


## Imports and paths

    This cell imports the model, diffusion, VAE, and sampling utilities, then defines the checkpoint/sample directories.


In [ ]:
import sys, json
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = Path.cwd()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))

from dit.configs import make_dit_config, RUNS
from dit.models import make_model
from dit.diffusion import GaussianDiffusion
from dit.ema import EMA
from dit.train_utils import load_checkpoint
from dit.latent import load_vae
from dit.sample_utils import save_class_grid

device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
CKPT_ROOT = PROJECT_ROOT / "checkpoints" / "latent_imagenette"
SAMPLE_ROOT = PROJECT_ROOT / "samples" / "latent_imagenette"
FIG_DIR = PROJECT_ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)
print("Device:", device)


## Choose a trained run

    This cell selects which checkpoint to sample. By default it uses `DiT-S-4/latest.pt`, the strongest run. If that checkpoint is missing, choose one of the Tiny runs that has completed.


In [ ]:
RUN_ID = "DiT-S-4"  # alternatives: DiT-Tiny-8, DiT-Tiny-4, DiT-Tiny-2
CKPT_PATH = CKPT_ROOT / RUN_ID / "latest.pt"
if not CKPT_PATH.exists():
    print("Checkpoint missing:", CKPT_PATH)
    print("Available checkpoints:", [p.parent.name for p in CKPT_ROOT.glob("*/latest.pt")])
    raise FileNotFoundError(CKPT_PATH)
ckpt = torch.load(CKPT_PATH, map_location="cpu")
print("Checkpoint step:", ckpt.get("step"))
if ckpt.get("step", 0) < 10_000:
    print("WARNING: checkpoint has fewer than 10k steps; samples may look undertrained.")
dit_cfg = make_dit_config(**{k: v for k, v in ckpt["dit_config"].items() if k in ["model_name", "patch_size", "input_size", "in_channels", "num_classes", "class_dropout_prob", "learn_sigma", "mlp_ratio", "conditioning"]})
print(dit_cfg)

## Load model, EMA weights, diffusion process, and VAE decoder

    This cell reconstructs the exact model from the checkpoint config. `USE_EMA=True` generally gives smoother samples after the runs.


In [ ]:
USE_EMA = True
model = make_model(dit_cfg).to(device)
diffusion = GaussianDiffusion(1000, device=device)
ema = EMA(model, decay=0.999)
_ = load_checkpoint(CKPT_PATH, model, ema=ema, device=device)
if USE_EMA and ckpt.get("ema") is not None:
    ema.apply_to(model)
    print("Using EMA weights.")
else:
    print("Using raw model weights.")
model.eval()
vae = load_vae(device)

## Generate a class-conditional sample grid

    This cell samples several images for each Imagenette class using DDIM and classifier-free guidance.


In [ ]:
CFG_SCALE = 1.5
DDIM_STEPS = 100
grid_path = SAMPLE_ROOT / RUN_ID / f"class_grid_step_{ckpt['step']:07d}_cfg_{CFG_SCALE}_ddim_{DDIM_STEPS}.png"
save_class_grid(model, diffusion, vae, grid_path, num_classes=10, samples_per_class=4, ddim_steps=DDIM_STEPS, cfg_scale=CFG_SCALE, device=device)
print("Saved:", grid_path)
img = Image.open(grid_path)
plt.figure(figsize=(8, 10))
plt.imshow(img)
plt.axis("off")
plt.title(f"{RUN_ID} samples, cfg={CFG_SCALE}, DDIM={DDIM_STEPS}")
plt.show()

## Classifier-free guidance sweep

    This cell shows how guidance changes sample appearance. Expected behavior: moderate guidance improves class alignment; too much guidance may reduce diversity or create artifacts.


In [ ]:
for cfg_scale in [1.0, 1.25, 1.5, 2.0]:
        out = SAMPLE_ROOT / RUN_ID / f"cfg_sweep_step_{ckpt['step']:07d}_cfg_{cfg_scale}.png"
        save_class_grid(model, diffusion, vae, out, num_classes=10, samples_per_class=2, ddim_steps=100, cfg_scale=cfg_scale, device=device, progress=False)
        print("Saved:", out)


## DDIM sampling-step sweep

    This cell tests whether more sampling steps improve visual quality. The paper argues that sampling compute has diminishing returns and cannot fully compensate for a weak model; this short experiment lets us inspect that behavior.


In [ ]:
for steps in [25, 50, 100, 250]:
        out = SAMPLE_ROOT / RUN_ID / f"step_sweep_step_{ckpt['step']:07d}_ddim_{steps}.png"
        save_class_grid(model, diffusion, vae, out, num_classes=10, samples_per_class=2, ddim_steps=steps, cfg_scale=1.5, device=device, progress=False)
        print("Saved:", out)
